In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

## Configure dependencies access

In [2]:
BASE_PATH = Path.cwd().parent.parent
ENV_PATH = BASE_PATH / "hospital_production" / ".env"

load_dotenv(ENV_PATH)

# PostgreSQL connection parameters
POSTGRES_USER: str = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD: str = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB: str = os.getenv("POSTGRES_DB")
POSTGRES_HOST: str = "localhost"
POSTGRES_PORT: str = os.getenv("POSTGRES_PORT")
POSTGRES_SCHEMA: str = os.getenv("POSTGRES_SCHEMA")
POSTGRES_TABLE: str = os.getenv("POSTGRES_TABLE")

# Build connection string
DATABASE_URL = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

# Create SQLAlchemy engine
engine = create_engine(DATABASE_URL)

# Open connection and execute query
with engine.connect() as conn:
    query = text(f"SELECT * FROM {POSTGRES_SCHEMA}.{POSTGRES_TABLE}")

    result = conn.execute(query)

    # Fetch and view the results into a pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

In [3]:
df.head()

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens,length_of_stay,patient_churn
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,3,3,27,66.0,27.0,39.0,63.0,72.0,25,0.0,25.0,6,1
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,12,2,40,26.0,26.0,0.0,96.0,56.0,25,23.0,2.0,1,0
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83,6,5,40,103.0,40.0,63.0,73.0,52.0,28,23.0,5.0,6,1
3,PAT-3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81,10,2,28,169.0,28.0,141.0,75.0,64.0,39,37.0,2.0,11,1
4,PAT-08591375,Garrett Lin,25,2025-02-18,2025-02-25,ICU,76,2,3,20,21.0,20.0,1.0,82.0,89.0,34,0.0,34.0,7,0


## Retrieve features from Feast

In [ ]:
import sys
from feast import FeatureStore

HOSPITAL_PRODUCTION = BASE_PATH / "hospital_production"

# Sys path to the project directory
sys.path.insert(0, str(HOSPITAL_PRODUCTION))
from app.ml.feast_store import FEATURES

# Create an entity dataframe with patient_id and arrival_date
entity_df = pd.read_sql(
    """
    SELECT patient_id, arrival_date AS event_timestamp
    FROM hospital.hospital_data
    """,
    engine,
)

# Store the features in a variables
store = FeatureStore(repo_path=str(HOSPITAL_PRODUCTION / "app" / "feast"))

# Get the historical feature values for the entity dataframe
features_df = store.get_historical_features(
        entity_df=entity_df,
        features=FEATURES,
    ).to_df()

# Rename the columns to remove the feature view name prefix
features_df.columns = [col.split(":")[-1]for col in features_df.columns]

# Show the features dataframe
display(features_df)

,patient_id,event_timestamp,age,satisfaction,length_of_stay,available_beds,patients_request,patients_admitted,patients_refused,staff_morale,staff_present,total_staff,staff_absent,capacity_pressure,admission_rate_row,refusal_rate_row,staff_attendance_rate,service
0,PAT-09484753,2025-03-16,24,61,6,27,66.00,27.00,39.00,72.00,0.00,25,25.00,2.357143,0.409091,0.590909,0.000000,surgery
1,PAT-f0644084,2025-12-13,6,83,1,40,26.00,26.00,0.00,56.00,23.00,25,2.00,0.634146,1.000000,0.000000,0.920000,surgery
2,PAT-ac6162e4,2025-06-29,24,83,6,40,103.00,40.00,63.00,52.00,23.00,28,5.00,2.512195,0.388350,0.611650,0.821429,general_medicine
3,PAT-3dda2bb5,2025-10-12,32,81,11,28,169.00,28.00,141.00,64.00,37.00,39,2.00,5.827586,0.165680,0.834320,0.948718,emergency
4,PAT-08591375,2025-02-18,25,76,7,20,21.00,20.00,1.00,89.00,0.00,34,34.00,1.000000,0.952381,0.047619,0.000000,ICU
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,PAT-bc982acf,2025-03-09,73,84,1,53,69.56,23.03,41.28,97.14,14.31,25,7.64,1.288148,0.331081,0.593445,0.572400,surgery
9996,PAT-94ea6637,2025-02-24,24,81,12,29,122.29,30.06,109.46,73.29,48.14,39,16.79,4.076333,0.245809,0.895085,1.234359,emergency
9997,PAT-042af740,2025-11-17,78,83,5,19,20.71,10.77,4.51,87.58,40.45,34,0.00,1.035500,0.520039,0.217769,1.189706,ICU
9998,PAT-6d3c54f9,2025-03-09,48,76,2,32,215.30,30.21,178.38,59.46,18.64,39,0.00,6.524242,0.140316,0.828518,0.477949,emergency
